### Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DataType
from pyspark.sql.functions import col,trim

In [0]:
%sql
select CNTRY from workspace.bronze.erp_loc_a101
group by CNTRY;

### Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")
print(f"print rows {df.count()}:")
display(df)

### Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))

### Customer Id cleanup

In [0]:
df = df.withColumn("cid",F.regexp_replace(col("cid"),"-",""))

### Country Normalization

In [0]:
df = df.withColumn(
    "CNTRY",
    F.when(col("CNTRY") == "DE","Germany")
     .when(col("CNTRY").isin("US","USA"),"United States")
     .when((col("CNTRY") == "") | col("CNTRY").isNull(),"n/a")
     .otherwise(col("CNTRY"))
)

### Renaming Columns
### 

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe

In [0]:
print("Sample data :")
df.limit(10).display()

print("\n Country distibution after Normalization :")
df.groupBy("country").count().orderBy("country",ascending=False).display()


### Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_loc_a101")
print(f"Written {df.count()} rows to workspace.silver.erp_loc_101")

In [0]:
%sql
select * from workspace.silver.erp_loc_a101